In [1]:
! uv pip install playwright
! playwright install

Using Python 3.12.12 environment at: C:\Workspace\LLM\llm_engineering\.venv
Checked 1 package in 10ms


In [2]:
import asyncio
from playwright.async_api import async_playwright
import nest_asyncio
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import requests
from openai import OpenAI

nest_asyncio.apply()

openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

class Website:
    title: str
    text: str
    url: str

    def __init__(self, url):
        self.url = url
         
    async def run(self, playwright):
        browser = await playwright.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(self.url)
        await page.wait_for_load_state('load')
        
        # Extract data from the page
        self.title = await page.title()
        text = await page.content()
        await browser.close()
    
        soup = BeautifulSoup(text, 'html.parser')
        for irrelevant in soup(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.get_text(separator="\n", strip=True)
    
    async def main(self):
        async with async_playwright() as playwright:
            await self.run(playwright)   
    
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

if __name__ == "__main__":
    site = Website('https://github.com/phedrakeson/draw/pull/10')
    asyncio.run(site.main())
    response = openai.chat.completions.create(
            model = "llama3.2",
            messages = messages_for(site)
        )

    web_summary = response.choices[0].message.content
    display(Markdown(web_summary))

Task exception was never retrieved
future: <Task finished name='Task-2' coro=<Connection.run() done, defined at c:\Workspace\LLM\llm_engineering\.venv\Lib\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\T-GAMER\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "c:\Workspace\LLM\llm_engineering\.venv\Lib\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "c:\Workspace\LLM\llm_engineering\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "c:\Workspace\LLM\llm_engineering\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

NotImplementedError: 